# Phase 1 (Day 2): The Vision Core - Image Data Extraction
### เป้าหมาย (Objective): สกัดข้อมูลดิบ (Raw Data) ทั้ง 3 มิติจากภาพโฆษณา ได้แก่ จำนวนคน (People Count), ข้อความ (Text), และสีหลัก (Dominant Colors)
### Objective: Extract three dimensions of raw data from the ad creatives: People Count, Overlay Text, and Dominant Colors.

## Step 1: Environment Setup & Library Installation
* การเตรียมพื้นที่ทำงานและติดตั้งเครื่องมือ:
เราเริ่มต้นด้วยการติดตั้งเครื่องมือจัดการแพ็กเกจ uv เพื่อให้การดาวน์โหลดและติดตั้งไลบรารี AI (YOLOv8, EasyOCR, OpenCV) ทำได้รวดเร็วกว่าการใช้ pip แบบดั้งเดิม
* Setting up the environment:
We started by installing the uv package manager to ensure lightning-fast installations of our core AI libraries (YOLOv8, EasyOCR, OpenCV), bypassing the traditional and slower pip.

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

downloading uv 0.10.6 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [ ]:
import os

# อัปเดต PATH ให้ Colab รู้จักคำสั่ง - uv Add 'uv' to PATH
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"

# uv pip install (ต้องเติม --system เพราะ Colab ไม่ได้ใช้ Virtual Env)
!uv pip install --system ultralytics opencv-python-headless pillow groq

Using Python 3.12.12 environment at: /usr
Audited 4 packages in 83ms


### Unzip PNG files

In [ ]:
!unzip -o ads_dataset.zip

Archive:  ads_dataset.zip
replace ads_dataset/ad_fashion_01.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
!ls

ads_dataset  ads_dataset.zip  sample_data  yolov8n.pt


## Step 2: Object Detection with YOLOv8
* การตรวจจับและนับจำนวนคนด้วย YOLOv8:
ใช้โมเดลคอมพิวเตอร์วิทัศน์ YOLO (You Only Look Once) รุ่นเล็ก yolov8n.pt เพื่อสแกนหาวัตถุในภาพ โดยเราเขียนเงื่อนไขกรองเฉพาะ Class 0 ซึ่งเป็นรหัสแทน "คน" (Person) เพื่อหาความหนาแน่นของบุคคลในโฆษณา
* Counting people using YOLOv8:
We utilized the lightweight computer vision model yolov8n.pt to scan for objects. A filter was applied to specifically isolate Class 0 (Person), allowing us to determine the human density within the ad creative.

In [ ]:
!pip install ultralytics opencv-python-headless pillow

In [ ]:
import cv2
from PIL import Image
from ultralytics import YOLO

# Initialize YOLOv8 Nano model
model = YOLO('yolov8n.pt')

# Define image path
img_path = 'ads_dataset/ad_fashion_03.png'

# Run inference
results = model(img_path)

# Filter and count only 'person'(ใน YOLO 'person' คือ class 0)
person_count = 0
for r in results:
    for box in r.boxes:
        if int(box.cls) == 0: # 0 คือ รหัสของ "คน"
            person_count += 1

print(f"People Count: {person_count}")

# Display the image with bounding boxes
res_plotted = results[0].plot()
Image.fromarray(res_plotted[:, :, ::-1])


image 1/1 /content/ads_dataset/ad_fashion_03.png: 352x640 2 persons, 123.1ms
Speed: 3.7ms preprocess, 123.1ms inference, 1.8ms postprocess per image at shape (1, 3, 352, 640)
People Count: 2


## Step 3: Optical Character Recognition (EasyOCR)
* การดึงข้อความจากภาพด้วย EasyOCR:
โฆษณาที่ดีต้องมีข้อความ (Copywriting) ที่ชัดเจน เราใช้ไลบรารี easyocr โหลดโมเดลภาษาอังกฤษ (en) เพื่ออ่านและสกัดข้อความ (Text Extraction) ที่ถูกฝังอยู่บนรูปภาพออกมาเป็นตัวอักษร
* Extracting text overlays with EasyOCR:
Effective ads rely on clear copywriting. We deployed the easyocr library with the English (en) language model to perform Text Extraction, converting embedded visual text into machine-readable strings.

In [ ]:
!uv pip install --system easyocr

Using Python 3.12.12 environment at: /usr
Audited 1 package in 127ms


In [ ]:
import easyocr

# Initialize EasyOCR reader for English
reader = easyocr.Reader(['en'])

# 2. Extract text from the image
ocr_results = reader.readtext(img_path)

print("--- Extracted Text ---")
extracted_texts = []
for (bbox, text, prob) in ocr_results:
    extracted_texts.append(text)
    print(f"Text: '{text}' (Confidence: {prob:.2f})")

## Step 4: Dominant Color Extraction (K-Means Clustering)
* การสกัดโทนสีหลักด้วยโมเดลคณิตศาสตร์ K-Means:
สีมีผลต่อจิตวิทยาการขาย เราใช้ OpenCV ย่อขนาดภาพเพื่อลดภาระการประมวลผล จากนั้นใช้ K-Means Clustering (จาก scikit-learn) เพื่อจัดกลุ่มพิกเซลสี และหาค่าสี RGB ที่โดดเด่นที่สุด 3 อันดับแรก พร้อมแปลงเป็นชื่อสีที่มนุษย์เข้าใจได้ผ่านสูตร Euclidean Distance
* Extracting Dominant Colors using K-Means:
Color heavily impacts consumer psychology. We used OpenCV to resize the image for optimization, then applied K-Means Clustering (via scikit-learn) to group color pixels and identify the top 3 dominant RGB values. Finally, we mapped these values to human-readable color names using Euclidean Distance.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans


def get_dominant_colors(image_path, k=3):
    # Loading pic and adjust color
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Resize for speed
    img = cv2.resize(img, (100, 100))
    img = img.reshape((-1, 3))

    # K-Means หาค่าสีหลัก
    clt = KMeans(n_clusters=k, n_init=10)
    clt.fit(img)

    return clt.cluster_centers_.astype(int)

def get_color_name(rgb):
    # Standard color database
    colors_db = {
        "Black": [0, 0, 0], "White": [255, 255, 255], "Gray": [128, 128, 128],
        "Dark Brown": [60, 40, 20], "Warm Brown": [150, 100, 50],
        "Beige/Cream": [230, 215, 190], "Navy Blue": [0, 0, 128],
        "Red": [200, 0, 0], "Gold": [212, 175, 55]
    }

    min_dist = float('inf')
    best_name = "Unknown"
    for name, value in colors_db.items():
        # Euclidean distance calculation
        dist = np.sqrt(np.sum((np.array(rgb) - np.array(value))**2))
        if dist < min_dist:
            min_dist = dist
            best_name = name
    return best_name

# Execution
colors = get_dominant_colors(img_path)
print("--- Dominant Colors ---")
for i, rgb in enumerate(colors):
    name = get_color_name(rgb)
    print(f"Color {i+1}: {name} (RGB: {rgb})")

## Phase 1 (Day 3): The LLM Brain
* เป้าหมาย (Objective): นำข้อมูลดิบที่สกัดได้จาก Computer Vision (จำนวนคน, ข้อความ, สี) ส่งให้โมเดลภาษาขนาดใหญ่ (LLM) วิเคราะห์ เพื่อประเมินคะแนนโฆษณาและแก้ไขข้อความที่ OCR อ่านผิดพลาด
* Objective: Pass the raw data extracted via Computer Vision (people count, text, colors) to a Large Language Model for analysis. The LLM will evaluate the ad creative and autocorrect OCR text hallucinations.

## Step 1: LLM Setup & Dependencies
#### 🛠️ Step 1 & 2: API Setup & Prompt Engineering
* การเชื่อมต่อ Groq API และบังคับโครงสร้างข้อมูล:**
เนื่องจากข้อจำกัดด้าน Data Privacy และโควตา API ในโซนยุโรป เราจึงสลับมาใช้ Groq API** ร่วมกับโมเดล **Llama 3.3 (70B)** ซึ่งมีความเร็วสูงและฉลาดเทียบเท่าโมเดลระดับท็อป เราใช้เทคนิค Prompt Engineering บังคับให้ AI ตอบกลับมาในรูปแบบ `JSON Object` เท่านั้น เพื่อให้ระบบทำงานต่อได้อย่างเสถียร
* API Integration and Structured Output:**
To bypass regional API quota restrictions in Europe, we pivoted to **Groq API** utilizing the **Llama 3.3 (70B)** model, known for its ultra-fast inference and high reasoning capabilities. We apply strict Prompt Engineering to enforce a `JSON Object` output, ensuring system stability for downstream data processing.

In [ ]:
import json
import os

from google.colab import userdata
from groq import Groq

# 1. ดึง API Key จาก Secrets
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    print("✅ Loading GROQ_API_KEY completed")
except userdata.SecretNotFoundError:
    print("❌ ยังไม่ได้ตั้งค่า GROQ_API_KEY ในช่อง Secrets ครับ!")

# 2. เริ่มต้นเชื่อมต่อ Groq
client = Groq()

## 🔑 Step 2: API Security & Structured Output (Pydantic Schema)
* การเชื่อมต่อ API และกำหนดโครงสร้างสมอง:**
เราตั้งค่า API Key ผ่านระบบ Secrets ของ Colab เพื่อความปลอดภัยสูงสุด (ไม่ Hardcode ลงในไฟล์) จากนั้นใช้ `pydantic.BaseModel` สร้าง "พิมพ์เขียว" (Schema) บังคับให้ AI ประเมินคะแนนโฆษณา (Rating 1-10), แก้ไขคำที่ OCR อ่านผิด (Autocorrect), และให้คำแนะนำทางธุรกิจ (Business Recommendation) ออกมาเป็น JSON เท่านั้น
* API Integration and Schema Definition:**
We securely inject the Groq API key using Colab's Secrets manager to prevent credential leakage. We then define a `pydantic.BaseModel` schema to force the LLM to output a strict JSON format containing an ad score (1-10), autocorrected OCR text, and actionable business recommendations.

##🧠 Step 3: The Master Prompt & API Execution
* การประกอบร่างและเรียกใช้ AI: เราจะจำลองข้อมูลดิบ (Mock Data) ที่สกัดได้จาก Computer Vision ใน Day 2 (เช่น จำนวนคน, ข้อความที่ OCR อ่านพลาด, และโทนสีหลัก) จากนั้นสร้าง Prompt เพื่อสั่งให้ AI สวมบทบาทเป็น Senior Creative Director ทำการวิเคราะห์ข้อมูลเหล่านี้ และส่งผลลัพธ์กลับมาในรูปแบบ JSON ตาม Schema ที่เรากำหนดไว้
* Prompt Engineering and API Execution: Construct a prompt that injects the raw data extracted from our Computer Vision pipeline (Day 2). We instruct AI to act as a Senior Creative Director, analyzing the raw inputs (e.g., hallucinated OCR text, dominant colors, human presence) to auto-correct errors and provide structured ad evaluation scores via the Pydantic schema.

In [ ]:
# 1. Mock Data: ข้อมูลที่สกัดได้จาก Day 2
extracted_data = {
    "person_count": 1,
    "raw_ocr_text": ["THE MASTER", "PAMEN"],
    "dominant_colors": ["Warm Brown", "Dark Brown", "Beige"]
}

# 2. Master Prompt
prompt = f"""
You are a Senior Creative Director evaluating a Japanese food advertisement.
Analyze the raw data and output ONLY a valid JSON object matching this exact structure:
{{
    "corrected_text": "autocorrect OCR text here",
    "design_score": integer between 1-10,
    "business_score": integer between 1-10,
    "actionable_feedback": "1-2 sentences of business advice"
}}

[Raw Data Extract]
- People in the ad: {extracted_data['person_count']}
- Dominant colors: {', '.join(extracted_data['dominant_colors'])}
- Text found in image (may contain OCR errors): {', '.join(extracted_data['raw_ocr_text'])}
"""

# 3. Call Groq API (lastest version)
print("⏳ Sending data to Groq API (Llama 3.3 70B)...")
try:
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        model="llama-3.3-70b-versatile", # 👈 เปลี่ยนตรงนี้ครับ
        response_format={"type": "json_object"},
        temperature=0.2
    )

    # 4. แสดงผลลัพธ์
    result_json = json.loads(response.choices[0].message.content)
    print("\n🎉 Result from Groq (Structured JSON):")
    print(json.dumps(result_json, indent=4, ensure_ascii=False))

except Exception as e:
    print(f"❌ mistake: {e}")